# Churn de telecomunicaciones: ejecucion secuencial

Ejecuta primero 00. Guarda y cierra 01-05. El maestro actualiza sus salidas y reportes, con un kernel nuevo por etapa. Se detiene si una celda falla. Cada llamada vuelve a preparar las particiones con las mismas reglas y semilla.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("telecom_churn")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from telecom_churn.alertas import configurar_notebook, mostrar_alertas
configurar_notebook()

TIMEOUT_POR_CELDA = 900


In [2]:
from telecom_churn.ejecucion import ejecutar_secuencia
if ejecutar_secuencia(PROJECT_ROOT, timeout=TIMEOUT_POR_CELDA):
    raise RuntimeError("Secuencia detenida. Revisa reports/ejecucion/resumen.json.")


Ejecutando 01_preparacion_datos.ipynb...
Completado 01_preparacion_datos.ipynb (1.97 s)
Ejecutando 02_analisis_exploratorio.ipynb...
Completado 02_analisis_exploratorio.ipynb (5.11 s)
Ejecutando 03_baseline.ipynb...
Completado 03_baseline.ipynb (1.95 s)
Ejecutando 04_modelado.ipynb...
Completado 04_modelado.ipynb (2.19 s)
Ejecutando 05_evaluacion.ipynb...
Completado 05_evaluacion.ipynb (2.39 s)
Los cinco notebooks terminaron sin errores de ejecución.


In [3]:
import json
import pandas as pd
resumen = json.loads((PROJECT_ROOT / "reports/ejecucion/resumen.json").read_text(encoding="utf-8"))
display(pd.DataFrame(resumen["notebooks"]))
from telecom_churn.informe import generar_informe
from IPython.display import Markdown
informe = generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


,notebook,estado,duracion_segundos
0,01_preparacion_datos.ipynb,completado,1.97
1,02_analisis_exploratorio.ipynb,completado,5.11
2,03_baseline.ipynb,completado,1.95
3,04_modelado.ipynb,completado,2.19
4,05_evaluacion.ipynb,completado,2.39


[Abrir informe integral](../reports/informe_final/informe_final.md)